# 01 - Extract Feature Vectors  (Stage 1)

**Purpose.** Read every Parcels Zarr store and, for each trajectory, record the
particle position 50 and 100 days after *its own* release:

`X_i = [lat_50, lon_50, lat_100, lon_100]`

plus the release date / month / year and a survival `status`.

**Inputs.**  ~1,628 `Parcels_run_*.zarr` stores (10,000 trajectories each)
listed by `config.list_stores()`.

**Output.**  `data/features.parquet` (one row per trajectory).

**Notes about this dataset (verified from the Zarr metadata):**
- Output frequency is **6-hourly**, time is absolute `datetime64` (NaT after a
  particle is deleted), longitude is already in -180..180, so no conversion.
- The number of obs **varies per store** (925 / 740 / ... and some are
  truncated to ~185 obs ≈ 46 days at the end of the forcing data). Day-50/100
  are therefore found by *time*, not a fixed index, and truncated stores fall
  out as `early_loss` / `partial` automatically.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import config as C
import pipeline as P
print("project root:", C.PROJECT_ROOT)

## 1.1  Configuration

In [ ]:
stores = C.list_stores()
print(f"{len(stores)} stores  ->  ~{len(stores)*C.TRAJ_PER_STORE:,} trajectories")
print("day-50 / day-100 offsets:", C.DAY_INTER, C.DAY_END, "| tolerance(d):", C.TIME_TOL_DAYS)
print("writing per-store parts to:", C.FEATURE_PARTS_DIR)
stores[:3]

## 1.2  Sanity check on a single store

`pipeline.extract_store_features` does the vectorised work: it converts each
obs time to *days since that particle's release*, finds the nearest valid obs to
day 50 and day 100, and assigns the status. Inspect one store first.

In [ ]:
df0 = P.extract_store_features(stores[0], 0)
print(df0.status.value_counts())
df0.head()

In [ ]:
# Quick scatter of the first 1000 day-50 vs day-100 positions as a sanity check.
import matplotlib.pyplot as plt
sub = df0[df0.status == "complete"].head(1000)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(sub.lon_50, sub.lat_50, s=4, label="day 50")
ax.scatter(sub.lon_100, sub.lat_100, s=4, label="day 100")
ax.set_xlabel("lon"); ax.set_ylabel("lat"); ax.legend(); ax.set_title("store 0 sample")
plt.show()

## 1.3  Process all stores

We loop store-by-store and write one small parquet per store into
`data/features_parts/`. This keeps memory flat (one store = 10k rows at a time)
and makes the loop restartable: already-written parts are skipped.

In [ ]:
import pyarrow as pa, pyarrow.parquet as pq

n = len(stores)
for i, s in enumerate(stores):
    part = C.FEATURE_PARTS_DIR / f"part_{i:05d}.parquet"
    if part.exists():
        continue
    df = P.extract_store_features(s, i)
    df.to_parquet(part, index=False)
    if (i + 1) % 50 == 0 or i == n - 1:
        print(f"Processing store {i+1}/{n} ...  ({s.name[:40]})")
print("done -> wrote", len(list(C.FEATURE_PARTS_DIR.glob('part_*.parquet'))), "parts")

## 1.4  Combine parts into a single `features.parquet`

In [ ]:
parts = sorted(C.FEATURE_PARTS_DIR.glob("part_*.parquet"))
features = pd.concat((pd.read_parquet(p) for p in parts), ignore_index=True)
features.to_parquet(C.FEATURES_FILE, index=False)
print("combined shape:", features.shape, "->", C.FEATURES_FILE)

## 1.5  Summary

In [ ]:
vc = features.status.value_counts()
print(f"Extracted features for {len(features):,} trajectories.")
for k in ["complete", "partial", "early_loss"]:
    print(f"  {k:10s}: {vc.get(k,0):,}")
print(f"release years: {features.release_year.min()}-{features.release_year.max()}")
print(f"Saved to {C.FEATURES_FILE}")